In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.gold;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import *

###dim_date

In [0]:
%sql
-- SELECT MIN(order_timestamp) AS min_order_date, MAX(order_timestamp) AS max_order_date FROM quickcart.silver.orders;
SELECT MIN(order_timestamp) AS min_delivery_order_date, MAX(actual_delivery_date) AS max_actual_delivery_date FROM quickcart.silver.deliveries;

In [0]:
start_date = "2025-07-01"
end_date = "2026-07-07"

date_df = spark.sql(f'''
                    SELECT EXPLODE(SEQUENCE(to_date('{start_date}'),
                    to_date('{end_date}'), interval 1 day)) AS full_date
                    ''')

dim_date_df = date_df.withColumn("date_sk", F.date_format("full_date","yyyyyMMdd").cast("int"))\
    .withColumn("day", F.dayofmonth("full_date"))\
    .withColumn("day_of_week", F.dayofweek("full_date"))\
    .withColumn("day_name", F.date_format("full_date", "EEEE"))\
    .withColumn("week_of_year", F.weekofyear("full_date"))\
    .withColumn("month", F.month("full_date"))\
    .withColumn("month_name", F.date_format("full_date", "MMMM"))\
    .withColumn("quarter", F.quarter("full_date"))\
    .withColumn("year", F.year("full_date"))\
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin([1, 7])
    )\
    .select(
        "date_sk",
        "full_date",
        "day",
        "day_of_week",
        "day_name",
        "week_of_year",
        "month",
        "month_name",
        "quarter",
        "year",
        "is_weekend"
    )\
    .orderBy("full_date")

display(dim_date_df)

In [0]:
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("quickcart.gold.dim_date")
)

In [0]:
%sql
SELECT *
FROM quickcart.gold.dim_date
ORDER BY full_date
LIMIT 20;

###dim_customer

In [0]:
%sql
SELECT customer_id, count(*)
from quickcart.silver.customers
group by customer_id
having count(*)>1
order by count(*) desc
limit 20;

In [0]:
%sql
SELECT COUNT(DISTINCT customer_id) AS unique_customer_ids
FROM quickcart.silver.customers;

In [0]:
df_customers = spark.table("quickcart.silver.customers")

window = Window.orderBy("customer_id")

dim_customers = df_customers.withColumn("customer_sk", F.row_number().over(window))\
    .withColumn("effective_from", F.col("updated_at"))\
        .withColumn("effective_to", F.lit(None).cast("timestamp"))\
            .withColumn("is_current", F.lit(True))\
                .select( "customer_sk",
        "customer_id",
        "customer_name",
        "email",
        "phone",
        "gender",
        "date_of_birth",
        "city",
        "state",
        "pincode",
        "registration_date",
        "customer_segment",
        "effective_from",
        "effective_to",
        "is_current")

In [0]:
dim_customers.write.format("delta").mode("overwrite").saveAsTable("quickcart.gold.dim_customers")

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM quickcart.gold.dim_customers;